# CirclePack in Jupyter — Demo

This notebook shows how to drive **CirclePack** from a Jupyter notebook,
freely mixing **Markdown**, **Python**, and **CirclePack commands** in one
document.

CirclePack runs as its own GUI application. This notebook talks to it over
its command socket using the `%%circlepack` cell magic (and the
`circlepack_client` Python module) that live in this folder.

> A *circle packing* is a configuration of circles with a prescribed pattern
> of tangencies; CirclePack computes and draws them. Here we'll make a few
> and view them inline.

## Prerequisites & setup

1. **Start CirclePack** so its command socket (port 3736) is listening —
   run the app jar from the folder that contains it:

   ```
   java -jar CirclePack-J5.2-jupyter.jar
   ```

2. **This notebook is already open in JupyterLab**, launched from this folder
   (so the `circlepack_*.py` helpers next to it can be imported).

Run the setup cells below **once per session**.

In [ ]:
%load_ext circlepack_magic

In [ ]:
%circlepack connect

### Inline images

`%%circlepack` cells display the packing by having CirclePack export an SVG to
its **packings** directory, which the magic reads back. That directory is
**auto-detected** (`./packings`, then `../packings`). Check it with `status`:

In [ ]:
%circlepack status

If `packdir` shows `None` (or images don't appear), set it by hand to the
folder CirclePack writes to:

```
%circlepack packdir C:\path\to\CirclePack\packings
```

## 1. Your first packing

A `%%circlepack` cell sends its lines to CirclePack as commands. Here we build
an *8-flower* (`seed 8`), compute its **maximal packing** (`max_pack`), and
draw it (`disp -w -c` = wipe canvas, draw circles).

> Tip: end a cell with a `disp …` command so the canvas — and the inline image
> — refresh.

In [ ]:
%%circlepack
seed 8
max_pack
disp -w -c

## 2. Colors

Color the boundary circles (`b`) and interior circles (`i`), then draw:

In [ ]:
%%circlepack
seed 12
max_pack
color -c 20 b
color -c 180 i
disp -w -c

## 3. Geometry: a hyperbolic maximal packing

CirclePack works in euclidean, hyperbolic, and spherical geometry. Convert to
hyperbolic (`geom_to_h`) and lay out the maximal packing inside the unit disk
(its boundary is drawn as well):

In [ ]:
%%circlepack
seed 10
geom_to_h
max_pack
disp -w -c

## 4. Mixing in Python

Because `%%circlepack` runs on the **ordinary Python kernel**, normal Python
cells work right alongside CirclePack cells.

In [ ]:
import math
sizes = [4, 6, 8, 12]
print("flower sizes:", sizes)
print("regular n-gon interior angle (deg):",
      [round(180 * (n - 2) / n, 1) for n in sizes])

### Driving CirclePack *from* Python

For loops or parameter sweeps, use the `circlepack_client` module directly.
Below we build several packings in a Python loop and collect the result codes.

In [ ]:
from circlepack_client import CirclePackClient

results = {}
with CirclePackClient(name="demo-python") as cp:
    for n in sizes:
        results[n] = cp.run(f"seed {n};max_pack")
results

## 5. Scripts ⇄ notebooks

A CirclePack script (`.cps`) is essentially a notebook already — explanatory
text interleaved with commands. The converters here translate both ways:

```
python cps2ipynb.py  some_script.cps  some_script.ipynb
python ipynb2cps.py  my_notebook.ipynb  my_script.cps
```

Everything you run here also appears in CirclePack's own **Command Terminal**
(*Advanced → Command Terminal*), where **Save as .cps** exports the session.

## Troubleshooting

- **`could not connect …`** — CirclePack isn't running, or not on port 3736.
  Start it, then re-run `%circlepack connect`.
- **No image appears** — run `%circlepack status`; if `packdir` is `None`, set
  it manually to CirclePack's packings folder.
- **Picture didn't change** — make sure the cell ends with a `disp …` command.
- **Query text** (e.g. `?rad 1`) prints in CirclePack's *own* window; the
  socket returns a result count, not message text.